In [ ]:
# Audio processing imports
import sys
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

# Import project code as src.*; putting src/ itself on the path would shadow Hugging Face `datasets`.
project_dir = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'datasets.py').exists())
sys.path.insert(0, str(project_dir))

# Dataset loading and inspection

In [ ]:
from src.audio.features import SETTINGS, SPLIT_NAMES, load_protocol
from src.datasets import dataset_path

la_dir = dataset_path('asvspoof19_la')
if la_dir is None or not la_dir.exists():
    raise FileNotFoundError('Set ASVSPOOF19_LA_DIR in .env to the ASVspoof 2019 LA folder.')

# Use protocol entries so every selected clip has a label.
splits = {}
for split in SPLIT_NAMES:
    splits[split], extra = load_protocol(la_dir, split)
    print(split, splits[split]['key'].value_counts().to_dict(), 'extra files:', extra)

train = splits['train']
examples = train.groupby('key').sample(n=5, random_state=42).copy()
info = examples['path'].map(sf.info)
examples['sample_rate'] = info.map(lambda item: item.samplerate)
examples['channels'] = info.map(lambda item: item.channels)
examples['duration'] = info.map(lambda item: item.duration)
display(examples[['sample_id', 'key', 'sample_rate', 'channels', 'duration']])

# Audio Processing

In [ ]:
# Settings in src/audio/features.py: 16 kHz mono, edge trim, peak normalization, 4 seconds.
from src.audio.features import process_audio

sample_rate = SETTINGS['sample_rate']

# Inspect one clip per class before extracting a batch.
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for row, key in enumerate(['bonafide', 'spoof']):
    path = examples.loc[examples['key'] == key, 'path'].iloc[0]
    original, _ = librosa.load(path, sr=sample_rate, mono=True)
    processed = process_audio(path)
    for ax, audio, title in zip(axes[row], [original, processed], ['Original', 'Processed']):
        ax.plot(np.arange(len(audio)) / sample_rate, audio)
        ax.set(title=f'{key}: {title}', xlabel='Time (seconds)', ylabel='Amplitude')
    print(key, path.stem)
    display(Audio(original, rate=sample_rate))
    display(Audio(processed, rate=sample_rate))
plt.tight_layout()
plt.show()

# MFCC baseline

20 coefficients with 25 ms windows and 10 ms hops. Mean and standard deviation over time produce 40 features per clip. Padding influences these statistics; keep settings fixed and compare trimmed/untrimmed audio later. This section checks feature extraction only, before training a classifier.

In [ ]:
from src.audio.features import mfcc_features, mfcc_matrix

mfcc = mfcc_matrix(process_audio(examples.iloc[0]['path']))
fig, ax = plt.subplots(figsize=(10, 4))
image = librosa.display.specshow(mfcc, sr=sample_rate, hop_length=SETTINGS['hop_length'], x_axis='time', ax=ax)
ax.set(title='MFCCs', ylabel='MFCC coefficient')
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

# Balanced smoke test, not the final training distribution.
subset = train.groupby('key').sample(n=50, random_state=42).copy()
X = mfcc_features(subset['path'])
labels = subset['label'].to_numpy()
sample_ids = subset['sample_id'].to_numpy()
assert X.shape == (100, 2 * SETTINGS['n_mfcc'])
assert np.isfinite(X).all()
print('MFCC matrix:', mfcc.shape)
print('Features:', X.shape, 'labels:', labels.shape)
print('Class counts:', subset['key'].value_counts().to_dict())

# Experiment settings and MFCC classifier

Default: balanced smoke subsets, not final research results. Set `FULL_RUN = True` for all protocol-listed clips. Validation selects thresholds; test and external labels never tune models. Outputs and caches stay under ignored `outputs/`. The earlier 100-clip cell is only an extraction check.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from src.audio.evaluate import new_classifier

FULL_RUN = False
seed = 42
run_name = 'full' if FULL_RUN else 'smoke'
output_dir = project_dir / 'outputs' / 'audio' / run_name
output_dir.mkdir(parents=True, exist_ok=True)
# Rename official train/dev/eval to the shared train/validation/test names.
frames = {SPLIT_NAMES[name]: frame.copy() if FULL_RUN else frame.groupby('key').sample(n=20, random_state=seed).copy()
          for name, frame in splits.items()}

mfcc_X = {name: mfcc_features(frame['path'], seed=seed) for name, frame in frames.items()}
targets = {name: frame['label'].to_numpy() for name, frame in frames.items()}
mfcc_model = new_classifier(seed)
mfcc_model.fit(mfcc_X['train'], targets['train'])
print(run_name, {name: values.shape for name, values in mfcc_X.items()})

# Pretrained encoder model

In [ ]:
import torch

from src.audio.encoder import cache_path, encoder_features, load_encoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor, encoder = load_encoder(device)

def embed(frame, cache_name=None, condition='clean'):
    # Clean fixed-window embeddings are cached; degraded conditions are recomputed.
    cache = cache_path(output_dir, cache_name) if cache_name else None
    return encoder_features(frame, processor, encoder, device, cache=cache, condition=condition, seed=seed)

encoder_X = {name: embed(frame, cache_name=name) for name, frame in frames.items()}
encoder_model = new_classifier(seed)
encoder_model.fit(encoder_X['train'], targets['train'])
print('Device:', device, 'Embedding shape:', encoder_X['train'].shape)

# In-Domain evaluation

In [ ]:
# Thresholds maximize validation F1; test labels never select them.
from src.audio.evaluate import evaluate_scores, select_threshold

models = {'mfcc': mfcc_model, 'wav2vec2': encoder_model}
feature_sets = {'mfcc': mfcc_X, 'wav2vec2': encoder_X}
scores = {}
thresholds = {}
results = []
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, models.items()):
    scores[name] = {split: model.predict_proba(values)[:, 1] for split, values in feature_sets[name].items()}
    thresholds[name] = select_threshold(targets['validation'], scores[name]['validation'])
    for split in ['validation', 'test']:
        results.append({'model': name, 'dataset': split, 'condition': 'clean',
                        **evaluate_scores(targets[split], scores[name][split], thresholds[name])})
    ConfusionMatrixDisplay(confusion_matrix(targets['test'], scores[name]['test'] >= thresholds[name]),
                           display_labels=['bonafide', 'spoof']).plot(ax=ax, colorbar=False)
    ax.set_title(f'{name}: test ({run_name})')
plt.tight_layout()
plt.show()
display(pd.DataFrame(results))

for name in models:
    fpr, tpr, _ = roc_curve(targets['test'], scores[name]['test'])
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.legend()
plt.title(f'Test ROC ({run_name})')
plt.show()

# Robustness and ablations
Compare clean-trained models on degraded test audio. Separately retrain MFCC baselines without trimming and with training-only noise/mu-law augmentation. Each retrained model selects its own threshold on matching clean validation data. Encoder trimming/augmentation retraining is not included. min t-DCF is not computed: it requires a matching ASV-score/cost protocol; AUC/EER alone are not substitutes.

In [ ]:
degraded_mfcc = {}
for condition in ['noise', 'mulaw']:
    degraded_mfcc[condition] = mfcc_features(frames['test']['path'], condition=condition, seed=seed)
    degraded_encoder = embed(frames['test'], condition=condition)
    for name, features in [('mfcc', degraded_mfcc[condition]), ('wav2vec2', degraded_encoder)]:
        prediction = models[name].predict_proba(features)[:, 1]
        results.append({'model': name, 'dataset': 'test', 'condition': condition,
                        **evaluate_scores(targets['test'], prediction, thresholds[name])})

untrimmed_X = {name: mfcc_features(frame['path'], trim=False, seed=seed) for name, frame in frames.items()}
untrimmed_model = new_classifier(seed).fit(untrimmed_X['train'], targets['train'])
untrimmed_threshold = select_threshold(targets['validation'], untrimmed_model.predict_proba(untrimmed_X['validation'])[:, 1])
prediction = untrimmed_model.predict_proba(untrimmed_X['test'])[:, 1]
results.append({'model': 'mfcc_untrimmed', 'dataset': 'test', 'condition': 'clean',
                **evaluate_scores(targets['test'], prediction, untrimmed_threshold)})

augmented_X = np.concatenate([mfcc_X['train']] + [mfcc_features(frames['train']['path'], condition=c, seed=seed)
                                                  for c in ['noise', 'mulaw']])
augmented_model = new_classifier(seed).fit(augmented_X, np.tile(targets['train'], 3))
augmented_threshold = select_threshold(targets['validation'], augmented_model.predict_proba(mfcc_X['validation'])[:, 1])
for condition, values in {'clean': mfcc_X['test'], **degraded_mfcc}.items():
    prediction = augmented_model.predict_proba(values)[:, 1]
    results.append({'model': 'mfcc_augmented', 'dataset': 'test', 'condition': condition,
                    **evaluate_scores(targets['test'], prediction, augmented_threshold)})
display(pd.DataFrame(results))

# External evaluation: In-the-Wild
Set `IN_THE_WILD_DIR` in `.env` to the directory containing `meta.csv` and the WAV files. Expected metadata columns: `file`, `speaker`, `label`, with labels `bona-fide` and `spoof`. Confirm this layout against your downloaded release. Missing data is reported as skipped, never as evaluated. No retraining or threshold tuning on external data.

In [ ]:
external_dir = dataset_path('in_the_wild')
external = None
external_scores = {}
if external_dir is None or not (external_dir / 'meta.csv').exists():
    print('External evaluation skipped: configure In-the-Wild data and meta.csv.')
else:
    external = pd.read_csv(external_dir / 'meta.csv')
    external['key'] = external['label'].astype(str).str.strip().map({'bona-fide': 'bonafide', 'bonafide': 'bonafide', 'spoof': 'spoof'})
    assert external['key'].notna().all(), 'Inspect unexpected external labels before evaluation'
    external['label'] = external['key'].map({'bonafide': 0, 'spoof': 1})
    external['sample_id'] = 'itw_' + external['file'].astype(str)
    external['path'] = [external_dir / str(name) for name in external['file']]
    assert external['sample_id'].is_unique
    assert external['path'].map(Path.exists).all()
    if not FULL_RUN:
        external = external.groupby('key').sample(n=20, random_state=seed).copy()
    external_X = {'mfcc': mfcc_features(external['path'], seed=seed), 'wav2vec2': embed(external, cache_name='external')}
    for name, model in models.items():
        external_scores[name] = model.predict_proba(external_X[name])[:, 1]
        results.append({'model': name, 'dataset': 'in_the_wild', 'condition': 'clean',
                        **evaluate_scores(external['label'], external_scores[name], thresholds[name])})
    display(pd.DataFrame(results).query("dataset == 'in_the_wild'"))

# Score export and conclusions
Higher score means higher spoof probability (`1 = spoof`, `0 = bonafide`). Training exports use out-of-fold predictions with scaling fitted inside each fold; validation/test use the final training-fitted model. These are sigmoid classifier probabilities, not independently calibrated risks. Fusion must align IDs using its pairing manifest; audio scores alone do not establish scam intent. Smoke exports are for integration checks only.

In [ ]:
import json

import joblib

folds = StratifiedKFold(n_splits=5 if FULL_RUN else 3, shuffle=True, random_state=seed)
for name, model in models.items():
    oof = cross_val_predict(new_classifier(seed), feature_sets[name]['train'], targets['train'],
                             cv=folds, method='predict_proba')[:, 1]
    tables = []
    for split, frame in frames.items():
        table = frame[['sample_id', 'label']].copy()
        table['split'] = split
        table['score'] = oof if split == 'train' else scores[name][split]
        tables.append(table[['sample_id', 'split', 'label', 'score']])
    pd.concat(tables, ignore_index=True).to_csv(output_dir / f'{name}_scores.csv', index=False)
    joblib.dump(model, output_dir / f'{name}_classifier.joblib')
    if external is not None:
        table = external[['sample_id', 'label']].copy()
        table['split'] = 'external'
        table['score'] = external_scores[name]
        table[['sample_id', 'split', 'label', 'score']].to_csv(output_dir / f'{name}_external_scores.csv', index=False)

metrics = pd.DataFrame(results)
metrics.to_csv(output_dir / 'metrics.csv', index=False)
settings = {'mode': run_name, 'seed': seed, **SETTINGS,
            'threshold_selection': 'maximum_validation_f1', 'thresholds': thresholds,
            'train_scores': 'stratified_out_of_fold', 'external_evaluated': external is not None}
(output_dir / 'settings.json').write_text(json.dumps(settings, indent=2))
for name in models:
    errors = frames['test'][['sample_id', 'key', 'attack']].copy()
    errors['score'] = scores[name]['test']
    errors['prediction'] = (errors['score'] >= thresholds[name]).astype(int)
    errors = errors.loc[errors['prediction'].to_numpy() != targets['test']]
    errors.to_csv(output_dir / f'{name}_test_errors.csv', index=False)
    print(name, 'test errors:', len(errors))
print('Saved:', output_dir)
display(metrics)

## Interpretation checklist after your full run

- Compare MFCC versus wav2vec 2.0 on test AUC, EER, and F1.
- Explain the trimmed/untrimmed difference and padding limitations.
- Compare clean/degraded performance and whether training augmentation helps.
- Report external generalization separately, or explicitly report missing data.
- Listen to representative false positives and false negatives from the error CSVs.
- Record that ASVspoof 2021 LA is optional and not implemented here; min t-DCF remains uncomputed.
- For full runs, set `FULL_RUN = True`, restart the kernel, and run all cells. Delete embedding caches after changing preprocessing or encoder settings. Smoke results do not support final project claims.